# SkinAI Image Audit and Training Splits

Copy these cells into your existing `image_audit.ipynb`, or use this notebook.
This notebook reads `manifests/master.parquet`; it never modifies that file,
your loader notebook, source metadata, or bucket. It writes manifests, not copies
of the images. Outputs still reference the source ZIP members.

Run the audit and review first. Export is deliberately gated by
`REVIEW_COMPLETE = False`. Do not change it until you inspect the reports.
Default project policy: keep DDI and MRA external; use approximately 80/10/10
train/validation/internal-test within the remaining usable data. Validation is
for tuning; neither internal nor external test is for model selection.

Dependencies, if needed in this kernel:
`%pip install pandas pyarrow pillow ImageHash huggingface_hub scikit-learn tqdm`
Authenticate Hugging Face using your existing local login. Never paste a token
into the notebook. Reading every image transfers substantially more data than
inspecting ZIP directories. Remote ZIP reads may be slow; checkpoints are saved
per archive and periodically within each archive.

## Configuration
Keep source archives unchanged during the audit. Increment AUDIT_VERSION after
any image/archive change, even if filenames and the master manifest are unchanged.
Reuse the same version to resume an interrupted run.

In [ ]:
from pathlib import Path
from contextlib import contextmanager
from io import BytesIO
import hashlib
import importlib.metadata
import json
import warnings
import zipfile

import pandas as pd
from PIL import Image, ImageFile, ImageOps
import imagehash
from huggingface_hub import HfFileSystem
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

PROJECT_ROOT = Path(r"C:\Users\12148\OneDrive\Documents\Capstone\Capstone460v2")
MASTER_PATH = PROJECT_ROOT / "manifests" / "master.parquet"
BUCKET = "sarcolemma/SkinAI"
AUDIT_VERSION = "v1"
SEED = 42
PHASH_DISTANCE = 4  # Candidate-screening policy, not a validated clinical threshold.
RETRY_FAILURES = False  # Set True to retry failed image reads on a resumed audit.
REVIEW_COMPLETE = True

CLASSES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
CLASS_TO_INDEX = {label: i for i, label in enumerate(CLASSES)}
EXTERNAL_SOURCES = {"ddi", "mra_midas"}
master_digest = hashlib.sha256(MASTER_PATH.read_bytes()).hexdigest()
OUT = PROJECT_ROOT / "manifests" / "image_audit" / f"{master_digest[:12]}_{AUDIT_VERSION}"
OUT.mkdir(parents=True, exist_ok=True)
master = pd.read_parquet(MASTER_PATH).sort_values("image_ref").reset_index(drop=True)
assert master["image_ref"].notna().all() and master["image_ref"].is_unique
assert not (set(master["label_class"].dropna()) - set(CLASSES))
fs = HfFileSystem(skip_instance_cache=True)
ImageFile.LOAD_TRUNCATED_IMAGES = False
print(f"Manifest: {len(master):,} rows; audit folder: {OUT}")

Manifest: 68,614 rows; audit folder: C:\Users\12148\OneDrive\Documents\Capstone\Capstone460v2\manifests\image_audit\73b6996b079f_v1


## Decode and hash every retained image

`Image.load()` loads pixel data; EXIF orientation is applied with
`ImageOps.exif_transpose()` before RGB conversion
([Pillow Image documentation](https://pillow.readthedocs.io/en/stable/reference/Image.html),
[Pillow ImageOps documentation](https://pillow.readthedocs.io/en/stable/reference/ImageOps.html)).
This audit rejects multiframe images for review rather than silently selecting a
frame. It records original mode, dimensions, byte MD5/SHA-256, canonical RGB pixel
SHA-256, and a 64-bit perceptual hash. A decoding success is not clinical quality
assurance; framing, labels, artifacts, and image appropriateness still need review.

`read_error` means reading from the ZIP failed, not that the image is corrupt.
Such errors block export until retried. `decode_error` and `multiframe` are
quarantined, subject to review. No failed source images are deleted.

In [ ]:
def remote_path(archive):
    return f"buckets/{BUCKET}/{archive}"

def archive_signature(archive):
    info = fs.info(remote_path(archive))
    return hashlib.sha256(
        json.dumps(info, sort_keys=True, default=str).encode()
    ).hexdigest()

@contextmanager
def open_archive(archive):
    with fs.open(remote_path(archive), "rb") as stream:
        with zipfile.ZipFile(stream) as z:
            yield z

def inspect_image(z, member, image_ref):
    record = {"image_ref": image_ref, "status": "read_error", "error": ""}
    try:
        data = z.read(member)  # Also checks ZIP CRC while reading this member.
    except Exception as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"
        return record
    record.update(byte_md5=hashlib.md5(data).hexdigest(),
                  byte_sha256=hashlib.sha256(data).hexdigest(), byte_count=len(data))
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", Image.DecompressionBombWarning)
            with Image.open(BytesIO(data)) as image:
                record.update(format=image.format, original_mode=image.mode,
                              frames=getattr(image, "n_frames", 1))
                if record["frames"] != 1:
                    record["status"] = "multiframe"
                    return record
                image.load()
                rgb = ImageOps.exif_transpose(image).convert("RGB")
                record.update(width=rgb.width, height=rgb.height)
                shape = f"RGB:{rgb.width}x{rgb.height}:".encode()
                record["pixel_sha256"] = hashlib.sha256(shape + rgb.tobytes()).hexdigest()
                record["phash"] = str(imagehash.phash(rgb, hash_size=8))
        record["status"] = "ok"
    except Exception as exc:
        record.update(status="decode_error", error=f"{type(exc).__name__}: {exc}")
    return record

refs = master["image_ref"].str.split("::", n=1, expand=True)
refs.columns = ["archive", "member"]
refs["image_ref"] = master["image_ref"]
archive_stamps = {}
audit_parts = []
for archive, group in refs.groupby("archive", sort=True):
    stamp = archive_signature(archive)
    archive_stamps[archive] = stamp
    checkpoint = OUT / (hashlib.sha256(archive.encode()).hexdigest()[:12] + ".parquet")
    old = pd.read_parquet(checkpoint) if checkpoint.exists() else pd.DataFrame()
    if not old.empty and not old["archive_signature"].eq(stamp).all():
        raise RuntimeError(f"{archive} changed: increment AUDIT_VERSION and start a fresh audit.")
    if not old.empty and RETRY_FAILURES:
        old = old.loc[old["status"].eq("ok")].copy()
    records = old.to_dict("records")
    done = set(old["image_ref"]) if not old.empty else set()
    pending = group.loc[~group["image_ref"].isin(done)]
    if not pending.empty:
        # Archive-open failures stop the cell rather than marking every image corrupt.
        with open_archive(archive) as z:
            for row in tqdm(pending.itertuples(index=False), total=len(pending), desc=archive):
                record = inspect_image(z, row.member, row.image_ref)
                record["archive_signature"] = stamp
                records.append(record)
                if len(records) % 250 == 0:
                    pd.DataFrame(records).to_parquet(checkpoint, index=False)
        pd.DataFrame(records).to_parquet(checkpoint, index=False)
    if archive_signature(archive) != stamp:
        raise RuntimeError(f"{archive} changed during the audit. Start a new audit version.")
    audit_parts.append(pd.DataFrame(records))

audit = pd.concat(audit_parts, ignore_index=True)
assert audit["image_ref"].is_unique and set(audit["image_ref"]) == set(master["image_ref"])
audit.to_parquet(OUT / "image_audit.parquet", index=False)
audit.to_csv(OUT / "image_audit.csv", index=False)
data = master.merge(audit, on="image_ref", how="left", validate="one_to_one")
data["source_split"] = data["split"]
data["is_external"] = (
    data["dataset_source"].isin(EXTERNAL_SOURCES)
    | data["source_split"].eq("external_test")
)
display(pd.crosstab(data["dataset_source"], data["status"]))
data.loc[data["status"].ne("ok")].to_csv(OUT / "unreadable_images.csv", index=False)
if not data["status"].eq("ok").any():
    raise RuntimeError("No readable images. Diagnose the audit before continuing.")

Images/Fitzpatrick17k.zip:  33%|███▎      | 5413/16577 [01:25<05:16, 35.23it/s] 'peer closed connection without sending complete message body (received 1224320 bytes, expected 5498385)' thrown while requesting GET https://huggingface.co/buckets/sarcolemma/SkinAI/resolve/Images/Fitzpatrick17k.zip
Retrying in 1s [Retry 1/5].
Images/milk10k.zip: 100%|██████████| 10480/10480 [01:04<00:00, 163.26it/s]


status,ok
dataset_source,
ddi,656
fitzpatrick17k,16577
hiba,1635
isic2019,33569
milk10k,10480
mra_midas,3399
pad_ufes_20,2298


## Exact duplicates and label conflicts

Exact-byte and identical canonical-pixel groups are reported separately.
The source `hash_code` is not overwritten: freshly computed hashes live in
separate columns. Supplied MD5 mismatches are reported for investigation.
For model exports, keep one row per canonical-pixel group, preferring an external
row, then a row with an eight-class label, then lexicographic image_ref.
Conflicting nonmissing target labels within identical pixels are quarantined;
this notebook never guesses which target label is correct.

In [ ]:
ok = data.loc[data["status"].eq("ok")].copy()
for column, filename in [
    ("byte_sha256", "exact_byte_duplicates.csv"),
    ("pixel_sha256", "exact_pixel_duplicates.csv"),
]:
    duplicates = ok.loc[ok[column].duplicated(keep=False)].sort_values([column, "image_ref"])
    duplicates.to_csv(OUT / filename, index=False)
    print(filename, len(duplicates), "rows")

conflicting_hashes = set(
    ok.groupby("pixel_sha256")["label_class"].nunique().loc[lambda n: n > 1].index
)
ok.loc[ok["pixel_sha256"].isin(conflicting_hashes)].to_csv(
    OUT / "exact_label_conflicts.csv", index=False
)
supplied = data["hash_code"].astype("string").str.strip().str.lower()
md5_mismatch = (
    data["status"].eq("ok")
    & supplied.str.fullmatch(r"[0-9a-f]{32}", na=False)
    & supplied.ne(data["byte_md5"]).fillna(False)
)
data.loc[md5_mismatch].to_csv(OUT / "source_md5_mismatches.csv", index=False)
print("Identical-pixel groups with conflicting labels:", len(conflicting_hashes))
print("Supplied MD5 mismatches:", int(md5_mismatch.sum()))

exact_byte_duplicates.csv 138 rows
exact_pixel_duplicates.csv 160 rows
Identical-pixel groups with conflicting labels: 15
Supplied MD5 mismatches: 0


## Screen visually similar images

Perceptual hashes screen for similarity, not proven duplicate identity
([ImageHash documentation](https://github.com/JohannesBuchner/imagehash/)).
We search all canonical-pixel representatives for pHash Hamming distance <= 4
using a small BK-tree, avoiding a full image-by-image matrix.
This threshold is a project choice. It can miss crops, rotations, overlays, and
other transformed copies, and can match unrelated images. It is not a guarantee
that all near-duplicates have been found.

Conservative policy: all candidate pairs stay in the SAME leakage group but
are NOT deleted merely for looking similar. This can over-group unrelated images
and distort split balance; inspect near_duplicate_candidates.csv, especially
large groups and cross-source pairs, before approving exports.
To reject false matches, put reviewed pairs in near_duplicate_rejections.csv
with columns image_ref_a,image_ref_b,reason; the next cell preserves that file.

In [ ]:
def nearby_hashes(tree, value, radius):
    if tree is None:
        return []
    found, stack = [], [tree]
    while stack:
        node = stack.pop()
        distance = (value ^ node[0]).bit_count()
        if distance <= radius:
            found.append(node[0])
        stack.extend(child for edge, child in node[1].items()
                     if distance - radius <= edge <= distance + radius)
    return found

def insert_hash(tree, value):
    if tree is None:
        return [value, {}]
    node = tree
    while value != node[0]:
        distance = (value ^ node[0]).bit_count()
        if distance not in node[1]:
            node[1][distance] = [value, {}]
            break
        node = node[1][distance]
    return tree

representatives = ok.drop_duplicates("pixel_sha256").sort_values("image_ref")
tree, seen, pairs = None, {}, []
for row in tqdm(representatives.itertuples(), total=len(representatives), desc="pHash screening"):
    value = int(row.phash, 16)
    for neighbor in nearby_hashes(tree, value, PHASH_DISTANCE):
        for previous in seen[neighbor]:
            pairs.append((previous, row.image_ref, (value ^ neighbor).bit_count()))
    seen.setdefault(value, []).append(row.image_ref)
    tree = insert_hash(tree, value)
    if len(pairs) > 250_000:
        raise RuntimeError("Too many candidate pairs. Inspect pHash collisions before continuing.")

near = pd.DataFrame(pairs, columns=["image_ref_a", "image_ref_b", "phash_distance"])
rejection_path = OUT / "near_duplicate_rejections.csv"
if not rejection_path.exists():
    pd.DataFrame(columns=["image_ref_a", "image_ref_b", "reason"]).to_csv(rejection_path, index=False)
rejections = pd.read_csv(rejection_path).fillna("")
if not rejections.empty and rejections["reason"].str.strip().eq("").any():
    raise ValueError("Every rejected similarity pair needs a review reason.")
rejected_pairs = {
    tuple(sorted((row.image_ref_a, row.image_ref_b)))
    for row in rejections.itertuples()
}
near["link_for_split"] = [
    tuple(sorted((a, b))) not in rejected_pairs
    for a, b in zip(near["image_ref_a"], near["image_ref_b"])
]
meta = data.set_index("image_ref")
for side in ["a", "b"]:
    for column in ["dataset_source", "label_class", "is_external"]:
        near[f"{column}_{side}"] = near[f"image_ref_{side}"].map(meta[column])
near.to_csv(OUT / "near_duplicate_candidates.csv", index=False)
print("Candidate pairs:", len(near), "| linked conservatively:", int(near["link_for_split"].sum()))
display(near.head(10))

pHash screening: 100%|██████████| 68534/68534 [02:53<00:00, 394.23it/s] 

Candidate pairs: 4282 | linked conservatively: 4278


,image_ref_a,image_ref_b,phash_distance,link_for_split,dataset_source_a,label_class_a,is_external_a,dataset_source_b,label_class_b,is_external_b
0,Images/DDI.zip::000291.png,Images/DDI.zip::000405.png,2,True,ddi,BCC,True,ddi,BCC,True
1,Images/DDI.zip::000292.png,Images/DDI.zip::000406.png,2,True,ddi,BCC,True,ddi,BCC,True
2,Images/DDI.zip::000293.png,Images/DDI.zip::000407.png,2,True,ddi,BCC,True,ddi,BCC,True
3,Images/DDI.zip::000297.png,Images/DDI.zip::000546.png,2,True,ddi,SCC,True,ddi,SCC,True
4,Images/Fitzpatrick17k.zip::Categorized_AbbrvNa...,Images/Fitzpatrick17k.zip::Categorized_AbbrvNa...,0,True,fitzpatrick17k,AK,False,fitzpatrick17k,AK,False
5,Images/Fitzpatrick17k.zip::Categorized_AbbrvNa...,Images/Fitzpatrick17k.zip::Categorized_AbbrvNa...,2,True,fitzpatrick17k,<NA>,False,fitzpatrick17k,<NA>,False
6,Images/Fitzpatrick17k.zip::Categorized_AbbrvNa...,Images/Fitzpatrick17k.zip::Categorized_AbbrvNa...,2,True,fitzpatrick17k,<NA>,False,fitzpatrick17k,<NA>,False
7,Images/Fitzpatrick17k.zip::Categorized_AbbrvNa...,Images/Fitzpatrick17k.zip::Categorized_AbbrvNa...,2,True,fitzpatrick17k,<NA>,False,fitzpatrick17k,<NA>,False
8,Images/Fitzpatrick17k.zip::Categorized_AbbrvNa...,Images/Fitzpatrick17k.zip::Categorized_AbbrvNa...,2,True,fitzpatrick17k,<NA>,False,fitzpatrick17k,<NA>,False
9,Images/Fitzpatrick17k.zip::Categorized_AbbrvNa...,Images/Fitzpatrick17k.zip::Categorized_AbbrvNa...,2,True,fitzpatrick17k,<NA>,False,fitzpatrick17k,<NA>,False


## Optional side-by-side inspection

Call `show_pair(0)` with a row number from the displayed candidate table.
Repeat for suspicious/cross-source pairs. This is a review aid, not an automatic
diagnosis. After adding false matches to the rejection CSV, rerun the screening,
grouping, and export cells. Do not edit candidate labels based only on appearance.

In [ ]:
def show_pair(pair_number):
    from IPython.display import display
    row = near.iloc[pair_number]
    for side in ["a", "b"]:
        ref = row[f"image_ref_{side}"]
        print(ref, "| label:", meta.at[ref, "label_class"])
        archive, member = ref.split("::", 1)
        with open_archive(archive) as z:
            with Image.open(BytesIO(z.read(member))) as image:
                preview = ImageOps.exif_transpose(image).convert("RGB")
                preview.thumbnail((480, 480))
                display(preview)

## Build leakage groups and retain eligible images

Connected components combine all available links: source-namespaced patient
and lesion IDs; global ISIC image IDs; fresh byte/pixel hashes; and non-rejected
visual-similarity candidates. Both patient and lesion links are applied, not one
instead of the other. Missing/sentinel IDs are not shared groups.

IDs from unrelated datasets are NOT assumed to identify the same patient.
Unknown patient/lesion identities, including unidentified cross-source links,
remain a limitation. Dermoscopic and clinical images are both retained, with
modality preserved for source/modality-specific evaluation.

Entire development-side groups connected to external data are quarantined,
rather than moving development-source images into the external benchmark.
Known-ID links are built even for unreadable rows, so these links are not lost.
No test-set membership is used to tune a model or choose model hyperparameters.

In [ ]:
parent = list(range(len(data)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def union(a, b):
    a, b = find(a), find(b)
    if a != b:
        parent[max(a, b)] = min(a, b)

def link_shared(keys):
    first = {}
    for i, key in keys.items():
        if pd.isna(key):
            continue
        if key in first:
            union(i, first[key])
        else:
            first[key] = i

for column in ["patient_id", "lesion_id"]:
    ids = data[column].astype("string").str.strip()
    ids = ids.mask(ids.str.lower().isin(["", "nan", "none", "null", "n/a", "unknown", "-1"]))
    link_shared(data["dataset_source"].astype("string") + ":" + column + ":" + ids)
global_isic = data["image_id"].astype("string").str.strip().str.upper()
link_shared(global_isic.where(global_isic.str.fullmatch(r"ISIC_\d+", na=False)))
for column in ["byte_sha256", "pixel_sha256"]:
    link_shared(data[column].where(data["status"].eq("ok")))
positions = dict(zip(data["image_ref"], data.index))
for row in near.loc[near["link_for_split"]].itertuples():
    union(positions[row.image_ref_a], positions[row.image_ref_b])
data["leakage_group"] = [f"g{find(i):07d}" for i in data.index]

data["exclusion_reason"] = pd.Series(pd.NA, index=data.index, dtype="string")
data.loc[data["status"].ne("ok"), "exclusion_reason"] = "image_audit_failure"
data.loc[data["pixel_sha256"].isin(conflicting_hashes), "exclusion_reason"] = "exact_label_conflict"
external_groups = set(data.loc[data["is_external"], "leakage_group"])
external_overlap = ~data["is_external"] & data["leakage_group"].isin(external_groups)
data.loc[external_overlap & data["exclusion_reason"].isna(), "exclusion_reason"] = "external_group_overlap"

available = data.loc[data["exclusion_reason"].isna()].copy()
available["labeled"] = available["label_class"].isin(CLASSES)
available = available.sort_values(
    ["is_external", "labeled", "image_ref"], ascending=[False, False, True]
)
redundant = available.index[available["pixel_sha256"].duplicated(keep="first")]
data.loc[redundant, "exclusion_reason"] = "redundant_exact_pixels"
clean = data.loc[data["exclusion_reason"].isna()].copy()
clean = clean.sort_values("image_ref").reset_index(drop=True)

data.loc[external_overlap].to_csv(OUT / "external_overlap.csv", index=False)
data.loc[data["exclusion_reason"].notna()].to_csv(OUT / "excluded_rows.csv", index=False)
group_sizes = data.groupby("leakage_group").agg(
    rows=("image_ref", "size"), sources=("dataset_source", "nunique"),
    patients=("patient_id", "nunique"), lesions=("lesion_id", "nunique")
).sort_values("rows", ascending=False)
group_sizes.to_csv(OUT / "leakage_group_sizes.csv")
data.to_parquet(OUT / "master_audited.parquet", index=False)
display(data["exclusion_reason"].fillna("retained").value_counts())
display(group_sizes.head(15))

exclusion_reason
retained                  68510
redundant_exact_pixels       65
exact_label_conflict         30
external_group_overlap        9
Name: count, dtype: int64[pyarrow]

,rows,sources,patients,lesions
leakage_group,,,,
g0003173,1458,5,34,670
g0002288,492,4,1,123
g0019489,361,1,0,59
g0019029,347,1,0,79
g0020118,45,1,0,12
g0024133,41,1,0,7
g0020127,32,1,0,7
g0038623,31,1,0,1
g0020681,28,1,0,6


## Review gate

Before proceeding, review unreadable_images.csv, exact_label_conflicts.csv,
source_md5_mismatches.csv, near_duplicate_candidates.csv, external_overlap.csv,
excluded_rows.csv, and the largest leakage groups. Byte-level read failures
must be retried, not accepted as corrupt images. Exact conflicts and decode
failures stay excluded unless you correct the inputs and rerun a fresh audit.

For unusual large groups, inspect IDs and candidate links. Reject confirmed
false similarity links using the rejection CSV and rerun from screening onward.
Do not discard legitimate patient links simply to improve class balance.

Set REVIEW_COMPLETE = True only after accepting these specific audit decisions.
Exported partitions are candidates for training, not a certification of clinical
validity or complete leakage elimination.

In [ ]:
if data["status"].eq("read_error").any():
    raise RuntimeError("Remote/member read errors remain. Set RETRY_FAILURES=True and rerun the audit.")
if not REVIEW_COMPLETE:
    raise RuntimeError("Review the saved audit reports, then set REVIEW_COMPLETE=True and rerun this cell.")
assert hashlib.sha256(MASTER_PATH.read_bytes()).hexdigest() == master_digest, "Master changed; start a fresh audit."

## Make splits once, then derive the eight-class subset

The selected policy uses 10 stratified GROUP folds: fold 0 is internal test,
fold 1 is validation, and the other eight folds are training. This is
approximately 80/10/10, not guaranteed exact percentages.
StratifiedGroupKFold attempts class balance while keeping groups non-overlapping;
large/uneven groups can prevent close stratification
([scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)).
DDI/MRA remain external_test. Out-of-scope/missing target labels receive a
temporary stratification category, never a ninth training class.

The eight-class subset inherits the SAME split assignments; it is not split
independently. The clean all-condition manifest is retained for other tasks,
but its missing target labels are not suitable for eight-class supervised loss.
Fit preprocessing, augmentation choices, encoders, and class weights using
training data only. Do not augment before splitting.

In [ ]:
if not REVIEW_COMPLETE or data["status"].eq("read_error").any():
    raise RuntimeError("Complete the audit review gate first.")
clean["split"] = "external_test"
development = clean.loc[~clean["is_external"]].copy()
strata = development["label_class"].astype("string").fillna("_OUT_OF_SCOPE_")
group_counts = development.assign(stratum=strata).groupby("stratum")["leakage_group"].nunique()
if (group_counts < 10).any():
    raise ValueError(
        "Fewer than 10 groups in a stratum. Review grouping/split policy; do not fall back to image-level splitting.\n"
        + group_counts.to_string()
    )
splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
for fold, (_, held_out) in enumerate(
    splitter.split(development, strata, groups=development["leakage_group"])
):
    assignment = "test" if fold == 0 else "val" if fold == 1 else "train"
    clean.loc[development.index[held_out], "split"] = assignment

assert clean.groupby("leakage_group")["split"].nunique().max() == 1
assert clean.groupby("pixel_sha256")["split"].nunique().max() == 1
assert clean.groupby("byte_sha256")["split"].nunique().max() == 1
assert clean.loc[clean["is_external"], "split"].eq("external_test").all()
assert not clean.loc[~clean["is_external"], "split"].eq("external_test").any()
eight = clean.loc[clean["label_class"].isin(CLASSES)].copy()
eight["class_index"] = eight["label_class"].map(CLASS_TO_INDEX).astype("int64")
for partition in ["train", "val", "test"]:
    absent = set(CLASSES) - set(eight.loc[eight["split"].eq(partition), "label_class"])
    if absent:
        raise ValueError(f"{partition} lacks classes {sorted(absent)}. Review groups and split design.")
display(pd.crosstab(eight["label_class"], eight["split"]))
display(pd.crosstab(clean["dataset_source"], clean["split"]))
display(pd.crosstab(eight["image_type"], eight["split"]))
display(pd.crosstab(eight["fitzpatrick_group"].fillna("unrecorded"), eight["split"]))

split,external_test,test,train,val
label_class,,,,
AK,193,244,1952,244
BCC,656,1110,8880,1110
BKL,326,475,3792,474
DF,73,58,458,58
MEL,256,762,6100,763
NV,718,1800,14393,1799
SCC,421,302,2416,302
VASC,58,64,512,63


split,external_test,test,train,val
dataset_source,,,,
ddi,656,0,0,0
fitzpatrick17k,0,1673,13233,1665
hiba,0,169,1349,115
isic2019,0,3337,26761,3411
milk10k,0,1054,8373,1048
mra_midas,3397,0,0,0
pad_ufes_20,0,214,1848,207


split,external_test,test,train,val
image_type,,,,
clinical,1936,457,3893,476
clinical: close-up,0,521,4094,516
dermoscopic,765,3837,30516,3821


split,external_test,test,train,val
fitzpatrick_group,,,,
I-II,135,0,0,0
III-IV,185,0,0,0
V-VI,84,0,0,0
unrecorded,2297,4815,38503,4813


## Save split manifests and reproducibility information

master.parquet/master.csv remain unchanged. In this audit-version folder:

- master_audited.parquet: every input row, audit flags, and leakage groups.
- master_clean_split.parquet/.csv: readable retained rows, all source diagnoses.
- predefined_8class.parquet/.csv: only MEL, NV, BCC, AK, BKL, DF, VASC, SCC.
- all_conditions/ and eight_classes/: train, val, test, external_test manifests.
- class_mapping.json and split_config.json: fixed target encoding and run details.
- split_counts.csv: source/class/modality/tone/split coverage for review.

Use eight_classes/train.parquet for eight-class training. Validation is for
tuning, internal test for final in-domain evaluation, and external_test for
source-held-out evaluation. Report DDI and MRA separately as well as any pooled
external result. Split ratios and subgroup coverage must be inspected before
freezing; do not repeatedly optimize the split using model test performance.

In [ ]:
def write_manifest(frame, destination):
    frame.to_parquet(destination.with_suffix(".parquet"), index=False)
    frame.to_csv(destination.with_suffix(".csv"), index=False)

write_manifest(clean, OUT / "master_clean_split")
write_manifest(eight, OUT / "predefined_8class")
for folder, frame in [("all_conditions", clean), ("eight_classes", eight)]:
    target = OUT / folder
    target.mkdir(exist_ok=True)
    for partition in ["train", "val", "test", "external_test"]:
        write_manifest(frame.loc[frame["split"].eq(partition)], target / partition)

coverage = eight.groupby(
    ["split", "dataset_source", "label_class", "image_type", "fitzpatrick_group"],
    dropna=False
).size().rename("rows").reset_index()
coverage.to_csv(OUT / "split_counts.csv", index=False)
(OUT / "class_mapping.json").write_text(json.dumps(CLASS_TO_INDEX, indent=2), encoding="utf-8")
config = {
    "master_sha256": master_digest, "audit_version": AUDIT_VERSION,
    "bucket": BUCKET, "archive_metadata_signatures": archive_stamps,
    "seed": SEED, "phash_hamming_threshold": PHASH_DISTANCE,
    "near_candidate_policy": "group unless explicitly rejected; never auto-delete",
    "split_policy": "8 train / 1 validation / 1 internal test group folds; external held out",
    "external_sources": sorted(EXTERNAL_SOURCES),
    "input_rows": len(data), "retained_rows": len(clean), "eight_class_rows": len(eight),
    "split_rows": {str(k): int(v) for k, v in eight["split"].value_counts().items()},
    "near_rejections_sha256": hashlib.sha256(rejection_path.read_bytes()).hexdigest(),
    "packages": {name: importlib.metadata.version(name) for name in
                 ["pandas", "Pillow", "ImageHash", "scikit-learn", "huggingface_hub", "pyarrow"]},
    "limitations": [
        "Unknown patient/lesion identities remain unverified, including across sources.",
        "pHash screening can miss transformed copies and can over-group unrelated images.",
        "Readable images are not necessarily clinically usable or correctly labelled.",
        "Class/source/modality/subgroup proportions are reported, not guaranteed balanced.",
    ],
}
(OUT / "split_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Saved reviewed split manifests to:", OUT)
print("Original master unchanged:", hashlib.sha256(MASTER_PATH.read_bytes()).hexdigest() == master_digest)

Saved reviewed split manifests to: C:\Users\12148\OneDrive\Documents\Capstone\Capstone460v2\manifests\image_audit\73b6996b079f_v1
Original master unchanged: True
